## 02_data_preparation_churn.ipynb
Este notebook realiza las siguientes tareas:

- Unificar ventas de 2022 a 2025
- Limpiar y estandarizar columnas
- Transformar la base transaccional a nivel de compra única
- Construir un dataset analítico a nivel cliente
- Definir churn de forma operativa

## 0. Librerías

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', None)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 1. Cargar datos

In [ ]:
RAW_PATH = Path('/content/drive/MyDrive/PI_LV/data/raw')
PROCESSED_PATH = Path('/content/drive/MyDrive/PI_LV/data/processed')
PROCESSED_PATH.mkdir(parents=True, exist_ok=True)

ventas_22_23 = RAW_PATH / "0_PI_Ventas-2022-2023 - ENVIAR.xlsx"
ventas_24_25 = RAW_PATH / "0_PI_Ventas-2024-2025 - ENVIAR.xlsx"

In [ ]:
ventas_2022 = pd.read_excel(ventas_22_23, sheet_name='Ventas_2022')
ventas_2023 = pd.read_excel(ventas_22_23, sheet_name='Ventas_2023')

ventas_2024 = pd.read_excel(ventas_24_25, sheet_name='Ventas_2024')
ventas_2025_1 = pd.read_excel(ventas_24_25, sheet_name='Ventas_2025-1')
ventas_2025_2 = pd.read_excel(ventas_24_25, sheet_name='Ventas_2025-2')

## 2. Normalizar columnas

In [ ]:
def normalize_columns(df):
    df = df.copy()
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace("-", "_", regex=False)
        .str.replace(" ", "_", regex=False)
    )
    return df

ventas_2022 = normalize_columns(ventas_2022)
ventas_2023 = normalize_columns(ventas_2023)
ventas_2024 = normalize_columns(ventas_2024)
ventas_2025_1 = normalize_columns(ventas_2025_1)
ventas_2025_2 = normalize_columns(ventas_2025_2)

## 3. Unir ventas

In [ ]:
ventas_total = pd.concat([
    ventas_2022,
    ventas_2023,
    ventas_2024,
    ventas_2025_1,
    ventas_2025_2
], ignore_index=True)

ventas_total.shape

(3973877, 31)

## 4. Seleccionar columnas clave

In [ ]:
ventas = ventas_total[[
    'documento',
    'documento_id_new',
    'fechaemision',
    'tipoventa',
    'terminopago',
    'terminopago_resumen',
    'vendedor_id',
    'gerencia_id',
    'unidadnegocio_id',
    'tipogerencia_id',
    'procedencia_id',
    'producto_id_new',
    'cliente_id_new',
    'domicilio_id',
    'cliente_dom_id_new',
    'indicador_impuesto',
    'moneda',
    'tipo_cambio',
    'descuento_porc',
    'almacen',
    'motivo_descuento',
    'fechavencimiento',
    'galones',
    'cantidad',
    'total_con_impuesto',
    'total_sin_impuesto',
    'total_costo',
    'total_promocion',
    'total_promocion_pv',
    'total_total_costo',
    'descuento'
]].copy()

## 5. Limpiar tipos

In [28]:
ventas['fechaemision'] = pd.to_datetime(ventas['fechaemision'], errors='coerce')
ventas['fechavencimiento'] = pd.to_datetime(ventas['fechavencimiento'], errors='coerce')

num_cols = [
    'tipo_cambio', 'descuento_porc', 'galones', 'cantidad',
    'total_con_impuesto', 'total_sin_impuesto', 'total_costo',
    'total_promocion', 'total_promocion_pv', 'total_total_costo',
    'descuento'
]

for col in num_cols:
    ventas[col] = pd.to_numeric(ventas[col], errors='coerce')

ventas = ventas.dropna(subset=['cliente_id_new', 'documento_id_new', 'fechaemision', 'total_con_impuesto']).copy()

# Mantener ventas positivas
ventas = ventas[ventas['total_con_impuesto'] > 0].copy()

ventas.shape

(2820890, 31)

## 6. Crear tabla de compra única

In [29]:
keys = ['cliente_id_new', 'documento_id_new', 'fechaemision']

flag_tipoventa = (
    ventas.groupby(keys)['tipoventa']
    .nunique()
    .reset_index(name='n_tipoventa')
)

flag_tipoventa['flag_tipoventa_inconsistente'] = (
    flag_tipoventa['n_tipoventa'] > 1
).astype(int)

Aquí consolidamos la información a nivel de cliente + documento + fecha, para que una fila represente una compra real y no una línea de producto.

In [30]:
compras = (
    ventas.groupby(keys, as_index=False)
    .agg({
        'total_con_impuesto': 'sum',
        'total_sin_impuesto': 'sum',
        'total_costo': 'sum',
        'descuento': 'sum',
        'descuento_porc': 'mean',
        'total_promocion': 'sum',
        'total_promocion_pv': 'sum',
        'cantidad': 'sum',
        'galones': 'sum',
        'producto_id_new': 'nunique',
        'unidadnegocio_id': 'first',
        'tipoventa': lambda x: x.mode().iloc[0] if not x.mode().empty else x.iloc[0],
        'terminopago': 'first',
        'terminopago_resumen': 'first',
        'vendedor_id': 'first',
        'gerencia_id': 'first',
        'tipogerencia_id': 'first',
        'procedencia_id': 'first',
        'motivo_descuento': 'first',
        'fechavencimiento': 'max'
    })
)

compras = compras.rename(columns={
    'producto_id_new': 'n_productos_compra'
})


In [31]:
compras = compras.merge(
    flag_tipoventa[['cliente_id_new', 'documento_id_new', 'fechaemision', 'flag_tipoventa_inconsistente']],
    on=['cliente_id_new', 'documento_id_new', 'fechaemision'],
    how='left'
)

compras.shape
compras.head()

,cliente_id_new,documento_id_new,fechaemision,total_con_impuesto,total_sin_impuesto,total_costo,descuento,descuento_porc,total_promocion,total_promocion_pv,cantidad,galones,n_productos_compra,unidadnegocio_id,tipoventa,terminopago,terminopago_resumen,vendedor_id,gerencia_id,tipogerencia_id,procedencia_id,motivo_descuento,fechavencimiento,flag_tipoventa_inconsistente
0,1027461,2024196089,2024-07-20,73.29,62.11,27.47,0.00000,0.0,0.0,0.0,1.0,0.7920,1,U116,Venta Neta,Contado,Contado,689,265,579,585,NINGUNO,2024-07-20,0
1,1027461,2024196090,2024-07-20,233.38,197.78,115.68,14.88690,7.0,0.0,0.0,6.0,36.0000,1,U116,Venta Neta,Contado,Contado,689,265,579,585,NINGUNO,2024-07-20,0
2,1027461,2024267952,2024-05-18,731.19,619.65,281.33,100.87392,14.0,0.0,0.0,5.0,30.0000,1,U116,Venta Neta,Contado,Contado,689,265,579,585,NINGUNO,2024-05-18,0
3,1027461,2024279382,2024-05-08,393.41,333.41,155.14,6.45864,2.0,0.0,0.0,6.0,31.9008,2,U116,Venta Neta,Contado,Contado,689,265,579,585,NINGUNO,2024-05-08,0
4,1027461,2025162235,2025-02-28,538.91,456.71,231.80,27.79500,5.0,0.0,0.0,6.0,5.6208,2,U116,Venta Neta,Contado,Contado,689,265,579,585,NINGUNO,2025-02-28,0


## 5.1. Variables auxiliares transaccionales

In [32]:
compras['year_month'] = compras['fechaemision'].dt.to_period('M')

# Rentabilidad
compras['margen_bruto'] = compras['total_sin_impuesto'] - compras['total_costo']
compras['margen_pct'] = np.where(
    compras['total_con_impuesto'] > 0,
    compras['margen_bruto'] / compras['total_sin_impuesto'],
    0
)

# Relación de descuentos y promociones sobre la venta
compras['ratio_descuento'] = np.where(
    compras['total_con_impuesto'] > 0,
    compras['descuento'].fillna(0) / compras['total_sin_impuesto'],
    0
)

compras['ratio_promocion'] = np.where(
    compras['total_con_impuesto'] > 0,
    compras['total_promocion'].fillna(0) / compras['total_sin_impuesto'],
    0
)

# Flags comerciales
compras['flag_descuento'] = (compras['descuento'].fillna(0) > 0).astype(int)
compras['flag_promocion'] = (compras['total_promocion'].fillna(0) > 0).astype(int)

# Condición de pago
compras['flag_credito'] = (
    compras['terminopago_resumen']
    .astype(str).str.strip().str.lower()
    .eq('credito')
    .astype(int)
)

compras['flag_contado'] = (
    compras['terminopago_resumen']
    .astype(str).str.strip().str.lower()
    .eq('contado')
    .astype(int)
)

# Plazo comercial
compras['dias_vencimiento'] = (
    compras['fechavencimiento'] - compras['fechaemision']
).dt.days

# Métricas unitarias
compras['precio_unitario_aprox'] = np.where(
    compras['cantidad'] > 0,
    compras['total_con_impuesto'] / compras['cantidad'],
    0
)

compras['precio_por_galon'] = np.where(
    compras['galones'] > 0,
    compras['total_con_impuesto'] / compras['galones'],
    0
)

# Fecha de referencia para recency
snapshot_date = compras['fechaemision'].max() + pd.Timedelta(days=1)

snapshot_date

Timestamp('2026-01-01 00:00:00')

## 5.2. Sustento del Umbral del Churn

In [33]:
intervalos = (
    compras.sort_values(['cliente_id_new', 'fechaemision'])
    .groupby('cliente_id_new')['fechaemision']
    .diff()
    .dropna()
    .dt.days
)

intervalos.describe(percentiles=[0.5, 0.75, 0.9, 0.95]).astype(int)

,fechaemision
count,1450196
mean,32
std,57
min,0
50%,14
75%,35
90%,75
95%,115
max,1454


El análisis de intervalos entre compras mostró que la mediana de recompra fue de 14 días, el percentil 75 fue de 35 días, el percentil 90 fue de 75 días y el percentil 95 fue de 115 días. En consecuencia, se consideró adecuado definir el **churn con un umbral de 70 días**, dado que este valor se aproxima al percentil 90 de la distribución y permite identificar clientes que se alejan significativamente del patrón normal de recompra

## 5.3. Función para construir RFM ampliado



In [34]:
def build_customer_features(group, snapshot_date):
    group = group.sort_values('fechaemision').copy()
    fechas = group['fechaemision']

    # Fechas clave
    fecha_primera = fechas.min()
    fecha_ultima = fechas.max()

    # RFM base
    recency = (snapshot_date - fecha_ultima).days
    frequency = len(group)
    monetary = group['total_con_impuesto'].sum()

    # Ciclo de vida y estabilidad
    tenure_days = (fecha_ultima - fecha_primera).days
    active_month_ratio = (
        fechas.dt.to_period('M').nunique() /
        max(
            1,
            (fecha_ultima.year - fecha_primera.year) * 12 + (fecha_ultima.month - fecha_primera.month) + 1
        )
    )

    # Intensidad monetaria y volumen
    avg_ticket = group['total_con_impuesto'].mean()
    std_ticket = group['total_con_impuesto'].std()
    total_galones = group['galones'].sum()

    # Intervalos entre compras
    diffs = fechas.diff().dropna().dt.days
    avg_days_between = diffs.mean() if len(diffs) > 0 else np.nan
    max_gap_days = diffs.max() if len(diffs) > 0 else np.nan

    # Actividad reciente alineada al churn
    last_70 = snapshot_date - pd.Timedelta(days=70)
    prev_70_start = snapshot_date - pd.Timedelta(days=140)
    prev_70_end = snapshot_date - pd.Timedelta(days=70)

    grp_last_70 = group[(group['fechaemision'] >= last_70) & (group['fechaemision'] < snapshot_date)]
    grp_prev_70 = group[(group['fechaemision'] >= prev_70_start) & (group['fechaemision'] < prev_70_end)]

    compras_70d = len(grp_last_70)
    gasto_70d = grp_last_70['total_con_impuesto'].sum()

    spend_last_70 = grp_last_70['total_con_impuesto'].sum()
    spend_prev_70 = grp_prev_70['total_con_impuesto'].sum()
    spend_trend_70 = (spend_last_70 - spend_prev_70) / (spend_prev_70 + 1)

    # Variables comerciales
    descuento_porc_promedio = group['descuento_porc'].mean()
    descuento_total = group['descuento'].sum()
    ratio_descuento_sobre_venta = descuento_total / (group['total_sin_impuesto'].sum() + 1e-6)
    margen_pct = group['margen_bruto'].sum() / (group['total_sin_impuesto'].sum() + 1e-6)

    # Condición comercial
    dias_promedio_vencimiento = group['dias_vencimiento'].mean()
    flag_credito = group['flag_credito'].mean()

    return pd.Series({
        'fecha_primera_compra': fecha_primera,
        'fecha_ultima_compra': fecha_ultima,
        'recency': recency,
        'frequency': frequency,
        'monetary': monetary,
        'tenure_days': tenure_days,
        'active_month_ratio': active_month_ratio,
        'avg_ticket': avg_ticket,
        'std_ticket': std_ticket,
        'total_galones': total_galones,
        'avg_days_between': avg_days_between,
        'max_gap_days': max_gap_days,
        'compras_70d': compras_70d,
        'gasto_70d': gasto_70d,
        'spend_trend_70': spend_trend_70,
        'descuento_porc_promedio': descuento_porc_promedio,
        'ratio_descuento_sobre_venta': ratio_descuento_sobre_venta,
        'margen_pct': margen_pct,
        'dias_promedio_vencimiento': dias_promedio_vencimiento,
        'flag_credito': flag_credito
    })

## 6. Agregar por cliente (RFM)

In [35]:
df_cliente = (
    compras
    .groupby('cliente_id_new')
    .apply(lambda g: build_customer_features(g, snapshot_date), include_groups=False)
)

df_cliente.shape
df_cliente.head()
df_cliente.isna().mean().sort_values(ascending=False)

/tmp/ipykernel_3547/3233978710.py:4: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: build_customer_features(g, snapshot_date))


,0
avg_days_between,0.154241
max_gap_days,0.154241
std_ticket,0.154241
fecha_primera_compra,0.000000
cliente_id_new,0.000000
frequency,0.000000
recency,0.000000
fecha_ultima_compra,0.000000
monetary,0.000000
avg_ticket,0.000000


Las variables asociadas a variabilidad temporal y dispersión del ticket presentan valores nulos en aproximadamente 15.4% de los clientes, lo cual se explica porque dichos clientes registran una sola compra o un historial insuficiente para calcular diferencias entre compras o desviación estándar.

## 7. Definir churn

In [37]:
df_cliente['churn_60'] = (df_cliente['recency'] > 60).astype(int)
df_cliente['churn_70'] = (df_cliente['recency'] > 70).astype(int)
df_cliente['churn_90'] = (df_cliente['recency'] > 90).astype(int)

In [38]:
df_cliente[['churn_60', 'churn_70', 'churn_90']].mean().to_frame('proporcion_churn')

,proporcion_churn
churn_60,0.593578
churn_70,0.564383
churn_90,0.530272


In [40]:
resumen_churn = pd.DataFrame({
    'Escenario': ['churn_60', 'churn_70', 'churn_90'],
    'No churn (0)': [
        df_cliente['churn_60'].value_counts().get(0, 0),
        df_cliente['churn_70'].value_counts().get(0, 0),
        df_cliente['churn_90'].value_counts().get(0, 0)
    ],
    'Churn (1)': [
        df_cliente['churn_60'].value_counts().get(1, 0),
        df_cliente['churn_70'].value_counts().get(1, 0),
        df_cliente['churn_90'].value_counts().get(1, 0)
    ],
    'Prop. No churn': [
        df_cliente['churn_60'].value_counts(normalize=True).get(0, 0),
        df_cliente['churn_70'].value_counts(normalize=True).get(0, 0),
        df_cliente['churn_90'].value_counts(normalize=True).get(0, 0)
    ],
    'Prop. Churn': [
        df_cliente['churn_60'].value_counts(normalize=True).get(1, 0),
        df_cliente['churn_70'].value_counts(normalize=True).get(1, 0),
        df_cliente['churn_90'].value_counts(normalize=True).get(1, 0)
    ]
}).round(4)

resumen_churn

,Escenario,No churn (0),Churn (1),Prop. No churn,Prop. Churn
0,churn_60,32658,47697,0.4064,0.5936
1,churn_70,35004,45351,0.4356,0.5644
2,churn_90,37745,42610,0.4697,0.5303


Se evaluaron tres escenarios alternativos de churn basados en recencia mayor a 60, 70 y 90 días. Los resultados mostraron proporciones de churn de 59.36%, 56.44% y 53.03%, respectivamente. Si bien la variación entre escenarios no fue extrema, se observó la tendencia esperada de disminución del churn al aumentar el umbral. Se seleccionó finalmente el umbral de 70 días, por estar mejor alineado con la distribución empírica de intervalos entre compras y ofrecer un balance razonable entre sensibilidad y criterio operativo.

In [41]:
df_cliente['churn'] = df_cliente['churn_70']

Revisar Churn

In [44]:
df_cliente['churn'].value_counts(normalize=True)

,proportion
churn,
1,0.564383
0,0.435617


In [46]:
df_cliente.shape

(80355, 25)

In [48]:
df_cliente.head()

,cliente_id_new,fecha_primera_compra,fecha_ultima_compra,recency,frequency,monetary,tenure_days,active_month_ratio,avg_ticket,std_ticket,total_galones,avg_days_between,max_gap_days,compras_70d,gasto_70d,spend_trend_70,descuento_porc_promedio,ratio_descuento_sobre_venta,margen_pct,dias_promedio_vencimiento,flag_credito,churn_60,churn_70,churn_90,churn
0,1027461,2024-05-08,2025-02-28,307,5,1970.18,296,0.300000,394.03600,256.588208,104.3136,74.000000,223.0,0,0.00,0.000000,5.600000,0.089847,0.514021,0.0,0.0,1,1,1,1
1,1089134,2025-02-06,2025-12-31,1,20,17891.83,328,1.000000,894.59150,687.827867,346.9040,17.263158,35.0,5,2555.38,0.069911,13.835327,0.175732,0.503229,0.0,0.0,0,0,0,0
2,1089187,2024-02-19,2025-12-08,24,31,7418.34,658,0.782609,239.30129,180.339686,220.0556,21.933333,129.0,3,680.83,-0.561248,11.661290,0.131741,0.440145,0.0,0.0,0,0,0,0
3,1111859,2024-10-16,2025-12-18,14,20,3583.75,428,0.866667,179.18750,99.518539,107.7428,22.526316,49.0,2,410.97,-0.490060,12.583333,0.162716,0.490562,0.0,0.0,0,0,0,0
4,1133970,2024-04-08,2025-12-04,28,8,937.10,605,0.333333,117.13750,72.922652,66.9528,86.428571,229.0,1,79.44,79.440000,8.625000,0.136548,0.369825,0.0,0.0,0,0,0,0


## 8. Limpieza final del dataset

In [49]:
df_cliente = df_cliente.replace([np.inf, -np.inf], np.nan)

num_cols_cliente = df_cliente.select_dtypes(include=[np.number]).columns
df_cliente[num_cols_cliente] = df_cliente[num_cols_cliente].fillna(0)

## 9. Guardar dataset analítico

In [50]:
df_cliente.to_csv(PROCESSED_PATH / 'clientes_churn.csv', index=False)